# W10D3 — LangGraph + Memory: Persistent Conversations

## Objective
Build and test a LangGraph workflow focused on persistent conversations.

## Tasks
1. Build a three-node graph: classify → route → respond.
2. Test five inputs and verify conditional routing.
3. Implement human-in-the-loop pause, human input, and resume.
4. Test persistent conversation memory.

## Technology Stack
- Python
- LangGraph StateGraph
- TypedDict
- InMemorySaver
- Jupyter Notebook

In [1]:
import sys
from importlib.metadata import version

print("Python executable:", sys.executable)
print("LangGraph version:", version("langgraph"))
print("LangChain Core version:", version("langchain-core"))

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

print("StateGraph: OK")
print("START/END: OK")
print("InMemorySaver: OK")

Python executable: c:\Users\ughan\CynarisInternship\AIML_PROJECT\.venv-w9d1\Scripts\python.exe
LangGraph version: 1.2.16
LangChain Core version: 1.6.13
StateGraph: OK
START/END: OK
InMemorySaver: OK


In [2]:
from typing import TypedDict


class AgentState(TypedDict):
    user_input: str
    category: str
    route: str
    response: str
    conversation_history: list[str]


print("State schema defined.")
print("State fields:", list(AgentState.__annotations__.keys()))

State schema defined.
State fields: ['user_input', 'category', 'route', 'response', 'conversation_history']


In [3]:
def classify_node(state: AgentState) -> dict:
    text = state["user_input"].lower()

    if any(word in text for word in ["error", "bug", "failed", "exception"]):
        category = "technical"
    elif any(word in text for word in ["hello", "hi", "hey", "good morning"]):
        category = "greeting"
    elif any(word in text for word in ["help", "how", "what", "explain", "why"]):
        category = "question"
    else:
        category = "general"

    return {"category": category}


sample_state = {
    "user_input": "My Python code has an error",
    "category": "",
    "route": "",
    "response": "",
    "conversation_history": [],
}

print("Classification result:", classify_node(sample_state)["category"])

Classification result: technical


In [4]:
def route_node(state: AgentState) -> dict:
    route_map = {
        "technical": "technical_support",
        "greeting": "friendly_greeting",
        "question": "answer_question",
        "general": "general_response",
    }

    category = state["category"]
    return {"route": route_map.get(category, "general_response")}


routing_state = {
    "user_input": "My Python code has an error",
    "category": "technical",
    "route": "",
    "response": "",
    "conversation_history": [],
}

routing_result = route_node(routing_state)

print("Category:", routing_state["category"])
print("Selected route:", routing_result["route"])

Category: technical
Selected route: technical_support


In [5]:
def respond_node(state: AgentState) -> dict:
    route = state["route"]
    user_input = state["user_input"]

    responses = {
        "technical_support": (
            f"I can help troubleshoot this technical issue: {user_input}"
        ),
        "friendly_greeting": "Hello! How can I help you today?",
        "answer_question": f"Let's work through your question: {user_input}",
        "general_response": f"Thanks for sharing. Tell me more about: {user_input}",
    }

    response = responses.get(
        route,
        "I'm ready to help. Please provide more details.",
    )

    # Preserve earlier conversation history and append this interaction.
    history = list(state.get("conversation_history", []))
    history.append(f"User: {user_input}")
    history.append(f"Assistant: {response}")

    return {
        "response": response,
        "conversation_history": history,
    }


response_state = {
    "user_input": "My Python code has an error",
    "category": "technical",
    "route": "technical_support",
    "response": "",
    "conversation_history": [],
}

response_result = respond_node(response_state)

print("Generated response:")
print(response_result["response"])
print("History entries:", len(response_result["conversation_history"]))

Generated response:
I can help troubleshoot this technical issue: My Python code has an error
History entries: 2


In [6]:
from langgraph.graph import StateGraph, START, END


def build_state_machine():
    builder = StateGraph(AgentState)

    builder.add_node("classify", classify_node)
    builder.add_node("route", route_node)
    builder.add_node("respond", respond_node)

    builder.add_edge(START, "classify")
    builder.add_edge("classify", "route")

    builder.add_conditional_edges(
        "route",
        lambda state: state["route"],
        {
            "technical_support": "respond",
            "friendly_greeting": "respond",
            "answer_question": "respond",
            "general_response": "respond",
        },
    )

    builder.add_edge("respond", END)

    return builder.compile()


state_machine = build_state_machine()

print("State machine compiled successfully.")
print("Application nodes:", ["classify", "route", "respond"])

State machine compiled successfully.
Application nodes: ['classify', 'route', 'respond']


In [7]:
test_inputs = [
    "Hello there!",
    "My Python code has an error",
    "How does a robot use computer vision?",
    "Explain LangGraph to me",
    "Today I worked on my project",
]

expected_categories = [
    "greeting",
    "technical",
    "question",
    "question",
    "general",
]

test_results = []

for user_input in test_inputs:
    result = state_machine.invoke({
        "user_input": user_input,
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    })

    test_results.append(result)

for index, (result, expected) in enumerate(
    zip(test_results, expected_categories), start=1
):
    passed = (
        result["category"] == expected
        and bool(result["route"])
        and bool(result["response"])
        and len(result["conversation_history"]) == 2
    )

    print(
        f"Test {index}: {'PASS' if passed else 'FAIL'} | "
        f"Expected={expected} | "
        f"Actual={result['category']} | "
        f"Route={result['route']} | "
        f"History entries={len(result['conversation_history'])}"
    )

passed_count = sum(
    result["category"] == expected
    and bool(result["route"])
    and bool(result["response"])
    and len(result["conversation_history"]) == 2
    for result, expected in zip(test_results, expected_categories)
)

print(f"\nResult: {passed_count}/5 tests passed")

Test 1: PASS | Expected=greeting | Actual=greeting | Route=friendly_greeting | History entries=2
Test 2: PASS | Expected=technical | Actual=technical | Route=technical_support | History entries=2
Test 3: PASS | Expected=question | Actual=question | Route=answer_question | History entries=2
Test 4: PASS | Expected=question | Actual=question | Route=answer_question | History entries=2
Test 5: PASS | Expected=general | Actual=general | Route=general_response | History entries=2

Result: 5/5 tests passed


In [9]:
from langgraph.checkpoint.memory import InMemorySaver

memory_saver = InMemorySaver()

builder = StateGraph(AgentState)

builder.add_node("classify", classify_node)
builder.add_node("route", route_node)
builder.add_node("respond", respond_node)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")

builder.add_conditional_edges(
    "route",
    lambda state: state["route"],
    {
        "technical_support": "respond",
        "friendly_greeting": "respond",
        "answer_question": "respond",
        "general_response": "respond",
    },
)

builder.add_edge("respond", END)

persistent_graph = builder.compile(checkpointer=memory_saver)

print("Persistent conversation graph compiled:", persistent_graph is not None)

Persistent conversation graph compiled: True


In [10]:
conversation_config = {
    "configurable": {"thread_id": "w10d3-conversation-1"}
}

# Turn 1
turn1 = persistent_graph.invoke(
    {
        "user_input": "Hello, my project is a smart agriculture robot.",
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    },
    config=conversation_config,
)

print("Turn 1 response:", turn1["response"])
print("Turn 1 history entries:", len(turn1["conversation_history"]))

# Turn 2: reuse the same thread_id
turn2 = persistent_graph.invoke(
    {
        "user_input": "Can you help me improve my project?",
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    },
    config=conversation_config,
)

print("\nTurn 2 response:", turn2["response"])
print("Turn 2 history entries:", len(turn2["conversation_history"]))
print("Previous message retained:", any(
    "smart agriculture robot" in item.lower()
    for item in turn2["conversation_history"]
))

Turn 1 response: Hello! How can I help you today?
Turn 1 history entries: 2

Turn 2 response: Let's work through your question: Can you help me improve my project?
Turn 2 history entries: 2
Previous message retained: False


In [11]:
def respond_node(state: AgentState) -> dict:
    route = state["route"]
    user_input = state["user_input"]

    responses = {
        "technical_support": (
            f"I can help troubleshoot this technical issue: {user_input}"
        ),
        "friendly_greeting": "Hello! How can I help you today?",
        "answer_question": f"Let's work through your question: {user_input}",
        "general_response": f"Thanks for sharing. Tell me more about: {user_input}",
    }

    response = responses.get(
        route,
        "I'm ready to help. Please provide more details.",
    )

    history = list(state.get("conversation_history", []))
    history.extend([
        f"User: {user_input}",
        f"Assistant: {response}",
    ])

    return {
        "response": response,
        "conversation_history": history,
    }


print("Response node updated to preserve conversation history.")

Response node updated to preserve conversation history.


In [12]:
memory_saver = InMemorySaver()

builder = StateGraph(AgentState)
builder.add_node("classify", classify_node)
builder.add_node("route", route_node)
builder.add_node("respond", respond_node)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")

builder.add_conditional_edges(
    "route",
    lambda state: state["route"],
    {
        "technical_support": "respond",
        "friendly_greeting": "respond",
        "answer_question": "respond",
        "general_response": "respond",
    },
)

builder.add_edge("respond", END)

persistent_graph = builder.compile(checkpointer=memory_saver)

print("Updated persistent graph compiled:", persistent_graph is not None)

Updated persistent graph compiled: True


In [13]:
conversation_config = {
    "configurable": {"thread_id": "w10d3-conversation-2"}
}

turn1 = persistent_graph.invoke(
    {
        "user_input": "My project is a smart agriculture robot.",
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    },
    config=conversation_config,
)

print("Turn 1 history entries:", len(turn1["conversation_history"]))

turn2 = persistent_graph.invoke(
    {
        "user_input": "Can you help me improve my project?",
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    },
    config=conversation_config,
)

print("Turn 2 history entries:", len(turn2["conversation_history"]))
print(
    "Previous message retained:",
    any(
        "smart agriculture robot" in item.lower()
        for item in turn2["conversation_history"]
    ),
)
print("\nConversation history:")
for item in turn2["conversation_history"]:
    print("-", item)

Turn 1 history entries: 2
Turn 2 history entries: 2
Previous message retained: False

Conversation history:
- User: Can you help me improve my project?
- Assistant: Let's work through your question: Can you help me improve my project?


In [14]:
saved_state = persistent_graph.get_state(conversation_config)

print("Checkpoint exists:", bool(saved_state.values))
print(
    "Saved history entries:",
    len(saved_state.values.get("conversation_history", [])),
)
print("Saved conversation history:")
for item in saved_state.values.get("conversation_history", []):
    print("-", item)

Checkpoint exists: True
Saved history entries: 2
Saved conversation history:
- User: Can you help me improve my project?
- Assistant: Let's work through your question: Can you help me improve my project?


In [15]:
from typing import Annotated, TypedDict
from operator import add


class AgentState(TypedDict):
    user_input: str
    category: str
    route: str
    response: str
    conversation_history: Annotated[list[str], add]


print("State schema updated with history reducer.")
print("History uses append semantics.")

State schema updated with history reducer.
History uses append semantics.


In [16]:
memory_saver = InMemorySaver()

builder = StateGraph(AgentState)

builder.add_node("classify", classify_node)
builder.add_node("route", route_node)
builder.add_node("respond", respond_node)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")

builder.add_conditional_edges(
    "route",
    lambda state: state["route"],
    {
        "technical_support": "respond",
        "friendly_greeting": "respond",
        "answer_question": "respond",
        "general_response": "respond",
    },
)

builder.add_edge("respond", END)

persistent_graph = builder.compile(checkpointer=memory_saver)

print("Graph rebuilt with history reducer:", persistent_graph is not None)

Graph rebuilt with history reducer: True


In [18]:
print("Turn 2 history entries:", len(turn2["conversation_history"]))
print("Previous message retained:", previous_message_retained)
print("\nActual history:")
for item in turn2["conversation_history"]:
    print("-", item)

print("\nCheckpoint history:")
saved_state = persistent_graph.get_state(conversation_config)
for item in saved_state.values.get("conversation_history", []):
    print("-", item)

Turn 2 history entries: 6
Previous message retained: True

Actual history:
- User: My project is a smart agriculture robot.
- Assistant: Thanks for sharing. Tell me more about: My project is a smart agriculture robot.
- User: My project is a smart agriculture robot.
- Assistant: Thanks for sharing. Tell me more about: My project is a smart agriculture robot.
- User: Can you help me improve my project?
- Assistant: Let's work through your question: Can you help me improve my project?

Checkpoint history:
- User: My project is a smart agriculture robot.
- Assistant: Thanks for sharing. Tell me more about: My project is a smart agriculture robot.
- User: My project is a smart agriculture robot.
- Assistant: Thanks for sharing. Tell me more about: My project is a smart agriculture robot.
- User: Can you help me improve my project?
- Assistant: Let's work through your question: Can you help me improve my project?


In [19]:
def respond_node(state: AgentState) -> dict:
    route = state["route"]
    user_input = state["user_input"]

    responses = {
        "technical_support": (
            f"I can help troubleshoot this technical issue: {user_input}"
        ),
        "friendly_greeting": "Hello! How can I help you today?",
        "answer_question": f"Let's work through your question: {user_input}",
        "general_response": f"Thanks for sharing. Tell me more about: {user_input}",
    }

    response = responses.get(
        route,
        "I'm ready to help. Please provide more details.",
    )

    return {
        "response": response,
        "conversation_history": [
            f"User: {user_input}",
            f"Assistant: {response}",
        ],
    }


print("Response node updated: emits only the current interaction.")

Response node updated: emits only the current interaction.


In [20]:

memory_saver = InMemorySaver()

builder = StateGraph(AgentState)
builder.add_node("classify", classify_node)
builder.add_node("route", route_node)
builder.add_node("respond", respond_node)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")
builder.add_conditional_edges(
    "route",
    lambda state: state["route"],
    {
        "technical_support": "respond",
        "friendly_greeting": "respond",
        "answer_question": "respond",
        "general_response": "respond",
    },
)
builder.add_edge("respond", END)

persistent_graph = builder.compile(checkpointer=memory_saver)

print("Persistent graph rebuilt:", persistent_graph is not None)

Persistent graph rebuilt: True


In [21]:
conversation_config = {
    "configurable": {"thread_id": "w10d3-conversation-final"}
}

turn1 = persistent_graph.invoke(
    {
        "user_input": "My project is a smart agriculture robot.",
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    },
    config=conversation_config,
)

turn2 = persistent_graph.invoke(
    {
        "user_input": "Can you help me improve my project?",
        "category": "",
        "route": "",
        "response": "",
        "conversation_history": [],
    },
    config=conversation_config,
)

history = turn2["conversation_history"]

print("Turn 1 history entries:", len(turn1["conversation_history"]))
print("Turn 2 history entries:", len(history))
print(
    "Previous message retained:",
    any("smart agriculture robot" in item.lower() for item in history),
)

for item in history:
    print("-", item)

assert len(history) == 4, f"Expected 4 entries, got {len(history)}"
assert any("smart agriculture robot" in item.lower() for item in history)

print("Persistent memory test: PASS")

Turn 1 history entries: 2
Turn 2 history entries: 4
Previous message retained: True
- User: My project is a smart agriculture robot.
- Assistant: Thanks for sharing. Tell me more about: My project is a smart agriculture robot.
- User: Can you help me improve my project?
- Assistant: Let's work through your question: Can you help me improve my project?
Persistent memory test: PASS


In [22]:
from langgraph.types import interrupt, Command


def human_review_node(state: AgentState) -> dict:
    decision = interrupt({
        "message": "Human review required",
        "draft_response": state["response"],
        "question": "Approve or replace this response?",
    })

    if isinstance(decision, dict):
        final_response = decision.get("response", state["response"])
    else:
        final_response = str(decision)

    return {"response": final_response}


print("Human-review node defined.")

Human-review node defined.


In [23]:
memory_saver = InMemorySaver()

builder = StateGraph(AgentState)

builder.add_node("classify", classify_node)
builder.add_node("route", route_node)
builder.add_node("respond", respond_node)
builder.add_node("human_review", human_review_node)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")

builder.add_conditional_edges(
    "route",
    lambda state: state["route"],
    {
        "technical_support": "respond",
        "friendly_greeting": "respond",
        "answer_question": "respond",
        "general_response": "respond",
    },
)

builder.add_edge("respond", "human_review")
builder.add_edge("human_review", END)

persistent_graph = builder.compile(checkpointer=memory_saver)

print("Human-review persistent graph compiled:", persistent_graph is not None)

Human-review persistent graph compiled: True


In [24]:
review_config = {
    "configurable": {"thread_id": "w10d3-human-review-1"}
}

review_input = {
    "user_input": "How can a robot use computer vision?",
    "category": "",
    "route": "",
    "response": "",
    "conversation_history": [],
}

paused_result = persistent_graph.invoke(
    review_input,
    config=review_config,
)

print("Graph paused for human review.")
print("Interrupt present:", bool(paused_result.get("__interrupt__")))

if paused_result.get("__interrupt__"):
    print("Review request:")
    print(paused_result["__interrupt__"][0].value)

Graph paused for human review.
Interrupt present: True
Review request:
{'message': 'Human review required', 'draft_response': "Let's work through your question: How can a robot use computer vision?", 'question': 'Approve or replace this response?'}


In [25]:
from langgraph.types import Command

human_decision = {
    "response": (
        "Robots use computer vision to detect objects, estimate their "
        "positions, recognize obstacles, and navigate their surroundings."
    )
}

resumed_result = persistent_graph.invoke(
    Command(resume=human_decision),
    config=review_config,
)

print("Graph resumed successfully.")
print("Final response:")
print(resumed_result["response"])
print(
    "Interrupt still pending:",
    bool(resumed_result.get("__interrupt__")),
)

Graph resumed successfully.
Final response:
Robots use computer vision to detect objects, estimate their positions, recognize obstacles, and navigate their surroundings.
Interrupt still pending: False


In [2]:
import json
from pathlib import Path

output_dir = Path("../outputs/W10D3")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation = {
    "assignment": "W10D3 - LangGraph Persistent Conversations",
    "classification_tests": {
        "total": 5,
        "passed": 5,
        "failed": 0,
        "status": "PASS",
    },
    "persistent_memory": {
        "turn1_history_entries": 2,
        "turn2_history_entries": 4,
        "previous_message_retained": True,
        "status": "PASS",
    },
    "human_review": {
        "interrupt_detected": True,
        "resumed_successfully": True,
        "interrupt_pending_after_resume": False,
        "status": "PASS",
    },
}

evaluation_path = output_dir / "evaluation_results.json"
evaluation_path.write_text(
    json.dumps(evaluation, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Evaluation saved:", evaluation_path.resolve())
print("Classification tests: 5/5")
print("Persistent memory: PASS")
print("Human-review pause/resume: PASS")
print("Evidence file exists:", evaluation_path.exists())

Evaluation saved: C:\Users\ughan\CynarisInternship\AIML_PROJECT\outputs\W10D3\evaluation_results.json
Classification tests: 5/5
Persistent memory: PASS
Human-review pause/resume: PASS
Evidence file exists: True


In [3]:
import json

with (output_dir / "evaluation_results.json").open(
    "r", encoding="utf-8"
) as file:
    saved_evaluation = json.load(file)

assert saved_evaluation["classification_tests"]["passed"] == 5
assert saved_evaluation["classification_tests"]["failed"] == 0
assert saved_evaluation["persistent_memory"]["previous_message_retained"] is True
assert saved_evaluation["human_review"]["resumed_successfully"] is True
assert saved_evaluation["human_review"]["interrupt_pending_after_resume"] is False

print("Saved JSON: valid")
print("Classification tests: 5/5 passed")
print("Persistent memory: PASS")
print("Human-review pause/resume: PASS")
print("Evidence verification: PASS")

Saved JSON: valid
Classification tests: 5/5 passed
Persistent memory: PASS
Human-review pause/resume: PASS
Evidence verification: PASS


In [4]:
validation_report = """# W10D3 Validation Report

## Assignment
LangGraph + Memory: Persistent Conversations

## Validation Results
- [x] Three-node graph: classify -> route -> respond
- [x] Conditional routing configured
- [x] Five classification tests passed (5/5)
- [x] Conversation history persisted across two turns
- [x] Previous message retained without duplicate entries
- [x] Human-review interrupt triggered
- [x] Human-provided response accepted
- [x] Graph resumed successfully
- [x] No interrupt remained after resume
- [x] Evaluation JSON verified

## Evidence
- `evaluation_results.json`

## Pending Workflow Checks
- [ ] CIA Full Stack Mentor Mode review
- [ ] At least two CIA interactions logged
- [ ] Two meaningful Git commits
- [ ] Branch pushed and pull request created
"""

validation_path = output_dir / "VALIDATION.md"
validation_path.write_text(validation_report, encoding="utf-8")

print("Validation report saved:", validation_path.resolve())
print("Report exists:", validation_path.exists())

Validation report saved: C:\Users\ughan\CynarisInternship\AIML_PROJECT\outputs\W10D3\VALIDATION.md
Report exists: True


In [5]:
import json

evaluation_path = output_dir / "evaluation_results.json"
validation_path = output_dir / "VALIDATION.md"

with evaluation_path.open("r", encoding="utf-8") as file:
    saved_evaluation = json.load(file)

assert saved_evaluation["classification_tests"]["passed"] == 5
assert saved_evaluation["persistent_memory"]["previous_message_retained"] is True
assert saved_evaluation["human_review"]["resumed_successfully"] is True
assert validation_path.exists()

print("Saved JSON: valid")
print("Classification tests: 5/5")
print("Persistent memory: PASS")
print("Human-review pause/resume: PASS")
print("Validation report: present")
print("Evidence verification: PASS")

Saved JSON: valid
Classification tests: 5/5
Persistent memory: PASS
Human-review pause/resume: PASS
Validation report: present
Evidence verification: PASS
